#### **<h1 align="center"><span style="color:#06445e;">Exploratory Data Analysis (EDA)</span>**
---
#### **<h1 align="center"><span style="color:#06445e;">About Author</span>**
### ***Project: The Impact of Promotions, Inflation, and Regional Conditions on Weekly Retail Sales***

### ***Author: Umair Ahmed***
### ***ERP: 28118***
### ***Email: u.ahmed.28118@khi.iba.edu.pk***

### ***Author: Aurad Asif***
### ***ERP: 28770***
### ***Email: m.asif.28770@khi.iba.edu.pk***



#### **<h1 align="center"><span style="color:#06445e;">About Data</span>**
### ***Title: Amazon Sales Dataset*** 
### ***Dataset:*** [Data Link](https://figshare.com/articles/dataset/amazon_sales_dataset_xlsx/28219025?file=51714707)
### **Source:** ***This dataset is from the website "Figshare"***

#### **<h1 align="center"><span style="color:#06445e;">Dataset Description</span>**
### ***The DataSet contains historical sales data for 45 Amazon stores located in different regions. Each store contains a number of departments, and have to predict the department-wide sales for each store.***<br><br>



#### **<h1 align="center"><span style="color:#06445e;">Dataset Column Description</span>**
#### **<h1 align="left"><span style="color:#06445e;">Features</span>**
### ***Store – The store number.***
### ***Dept – The department number.***
### ***Date – The week of the observation.***
### ***Weekly_Sales – Sales for the given department in the given store.***
### ***IsHoliday – Indicates whether the week is a special holiday week.***
### ***Temperature – Average temperature in the region.***
### ***Fuel_Price – Cost of fuel in the region.***
### ***Total_MarkDown – Anonymized data related to promotional markdowns that Amazon is running. Markdown data is only available after November 2020 and may not be available for all stores at all times. Any missing value is marked as NA***
### ***CPI – Consumer Price Index.***
### ***Unemployment – Unemployment rate.***<br><br>


#### **<h1 align="center"><span style="color:#06445e;">Kernel Version Used</span>**
- ***Python 3.12.7***

#### **<h1 align="center"><span style="color:#06445e;">Importing Libraries</span>**

In [ ]:
# Import libraries and configure plotting and warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew
from scipy.stats import gaussian_kde
from math import sqrt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly import tools
from plotly.offline import init_notebook_mode, iplot
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings

sns.set()
%matplotlib inline
init_notebook_mode(connected=True)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

#### **<h1 align="center"><span style="color:#06445e;">Data Exploration & Cleaning</span>**

### **1.1 Load Data**

In [ ]:
# Load the raw sales dataset
df = pd.read_excel("amazon_sales_dataset.xlsx")

### **1.2  Studying and Understanding the Data**


In [ ]:
# Preview the first few rows
df.head()

In [ ]:
# Check the dataset's dimensions and column names
print(df.shape)

print(df.columns)

In [ ]:
# Inspect data types and non-null counts
df.info();
print(f"\n * df has {df.shape[1]} columns and {df.shape[0]} rows.");

In [ ]:
# Cast flags and IDs to readable, memory-efficient types
df["IsHoliday"] = df["IsHoliday"].astype(bool)
df["IsHoliday"] = df["IsHoliday"].map({True: "Yes", False: "No"})
df["Store"] = df["Store"].astype("category")
df["Dept"] = df["Dept"].astype("category")
df["Type"] = df["Type"].astype("category")

In [ ]:
# Rename Total_MarkDown to a clearer business name
df.rename(columns={'Total_MarkDown': 'Promotional_Discount'}, inplace=True)

In [ ]:
# Preview after the rename
df.head()

In [ ]:
# Summary statistics for the numeric columns
df.describe()

In [ ]:
# Summary of the categorical columns
df.describe(include=[object, "category"])

### 1.2.1 Columns by types

In [ ]:
# Group columns into numeric and categorical lists
num_cols = ['Weekly_Sales','Size','Temperature','Fuel_Price','CPI','Unemployment','Promotional_Discount']
cat_cols = ['Store','Dept','Type','IsHoliday','Year','Month','Week']

### Numerical Columns Distribution:

In [ ]:
# Plot a distribution and box plot for each numeric feature
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import math

n = len(num_cols)
cols = 2
rows = n

subplot_titles = []
for col in num_cols:
    subplot_titles.append(f"{col} Distribution")
    subplot_titles.append(f"{col} Summary Stats")

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=subplot_titles
)

for idx, col in enumerate(num_cols):
    row = idx + 1

    fig.add_trace(
        go.Histogram(x=df[col], nbinsx=30, name="", showlegend=False),
        row=row, col=1
    )

    fig.add_trace(
        go.Box(x=df[col], boxmean="sd", name="", showlegend=False),
        row=row, col=2
    )

fig.update_layout(
    height=300 * rows,
    width=1000,
    title_text="Numerical Feature Distributions and Summary Statistics",
    showlegend=False
)

fig.show()

### Categorical Columns Plot:

In [ ]:
# Plot the category counts for each categorical feature
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import math

n = len(cat_cols)
cols = 2
rows = math.ceil(n / cols)

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=cat_cols
)

for i, col in enumerate(cat_cols):
    row = i // cols + 1
    col_pos = i % cols + 1
    counts = df[col].value_counts()

    fig.add_trace(
        go.Bar(
            x=counts.index,
            y=counts.values,
            text=counts.values,
            textposition="inside",
            name=col
        ),
        row=row,
        col=col_pos
    )

fig.update_layout(
    height=300 * rows,
    width=1000,
    title_text="Categorical Feature Distributions",
    showlegend=False
)

fig.show()

### **1.3 Data Cleaning**

In [ ]:
# Preview the rows
df.head(10)

In [ ]:
# Rebuild Week by numbering the weeks within each month
unique_dates = df['Date'].sort_values().unique()
week_numbers = []
for (year, month), group in df.groupby(['Year', 'Month']):
    unique_dates = sorted(group['Date'].unique())
    date_to_week = {date: i+1 for i, date in enumerate(unique_dates)}
    week_numbers.extend([date_to_week[date] for date in group['Date']])
df['Week'] = week_numbers
df.head()

In [ ]:
# Drop pre-computed stat columns that add nothing to the analysis
df = df.drop(columns=["min", "max", "mean", "median", "std"], errors='ignore')

### 1.3.1 Check Duplicates

In [ ]:
# Check for duplicate rows
print(f"In the data set duplicates are {df.duplicated()} .\n")
print(f"The data set has {df.duplicated().sum()} duplicates.\n")

In [ ]:
# Drop the unnamed column that just duplicates Date
df = df.drop(columns=["Unnamed: 3"], errors='ignore')

In [ ]:
# Preview the rows
df.head()

### 1.3.2 Check Missing Values

In [ ]:
# Count the total missing values
total_nulls = df.isna().sum().sum()
print(f"Total number of NaN values in the dataset: {total_nulls}")

In [ ]:
# Count the zero values in each numeric column
zero_counts = (df[num_cols] == 0).sum()
print(zero_counts)

In [ ]:
# Find the first week that markdown data appears (Nov 2020)
nov_2020 = df[(df['Year'] == 2020) & (df['Month'] == 11)]
available_data = nov_2020[nov_2020['Promotional_Discount'] != 0]
first_week_available = available_data['Week'].min()
print(f"The first week in November 2020 with markdown data available is Week {first_week_available}.")

In [ ]:
# Confirm there are no genuine zeros once markdowns are reported
zeroes_after_2020 = (df[df['Year'] > 2020]['Promotional_Discount'] == 0).any()
print(zeroes_after_2020)

In [ ]:
# Missing markdowns are unreported values, not true zeros

In [ ]:
# Impute missing markdowns with each store's post-2020 average
store_avg = df[df['Year'] > 2020].groupby('Store')['Promotional_Discount'].mean().to_dict()

df['Promotional_Discount'] = df.apply(
    lambda x: store_avg[x['Store']] if x['Promotional_Discount'] == 0 else x['Promotional_Discount'],
    axis=1)

zero_count = (df['Promotional_Discount'] == 0).sum()
print(f"Number of zeros remaining: {zero_count}")

In [ ]:
# Impute zero weekly sales with the store/dept/month average
store_dept_month_avg = df[df['Weekly_Sales'] != 0].groupby(
    ['Store','Dept','Month','Type']
)['Weekly_Sales'].mean().to_dict()

df['Weekly_Sales'] = df.apply(
    lambda x: store_dept_month_avg.get((x['Store'], x['Dept'], x['Month']), x['Weekly_Sales']) 
              if x['Weekly_Sales'] == 0 else x['Weekly_Sales'],
    axis=1
)

zero_countss = (df['Weekly_Sales'] == 0).sum()
print(f"Number of zeros remaining: {zero_countss}")

In [ ]:
# Re-check the remaining zeros per numeric column
zero_counts = (df[num_cols] == 0).sum()
print(zero_counts)

### 1.3.3 Dealing with Outliers

In [ ]:
# Define an IQR-based outlier detector
target_cols = ['Weekly_Sales', 'Promotional_Discount', 'Temperature', 'Fuel_Price']

def detect_iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series[(series < lower) | (series > upper)].index

In [ ]:
# Check the shape
df.shape

In [ ]:
# Replace outliers in the key columns with the column median
df_imputed = df.copy()

for col in target_cols:
    outliers = detect_iqr_outliers(df_imputed[col])
    median = df_imputed[col].median()
    df_imputed.loc[outliers, col] = median
df_imputed.shape

In [ ]:
# Preview the rows
df.head()

### **1.4 Creating Bins**

In [ ]:
# Bin promotional discount into intensity levels
df["Promotion_Bin"] = "No Promotion"

positive_md = df.loc[df["Promotional_Discount"] > 0, "Promotional_Discount"]

q25, q50, q75 = positive_md.quantile([0.25, 0.50, 0.75])

df.loc[(df["Promotional_Discount"] > 0) & (df["Promotional_Discount"] <= q25), "Promotion_Bin"] = "Low Promotion"
df.loc[(df["Promotional_Discount"] > q25) & (df["Promotional_Discount"] <= q50), "Promotion_Bin"] = "Medium Promotion"
df.loc[(df["Promotional_Discount"] > q50) & (df["Promotional_Discount"] <= q75), "Promotion_Bin"] = "High Promotion"
df.loc[df["Promotional_Discount"] > q75, "Promotion_Bin"] = "Very High Promotion"

df["Promotion_Bin"] = df["Promotion_Bin"].astype("category")

In [ ]:
# Preview the rows
df.head()

In [ ]:
# Bin CPI into inflation levels
cpi_33, cpi_66 = df["CPI"].quantile([0.33, 0.66])

df["Inflation_Bin"] = "Low Inflation"
df.loc[(df["CPI"] > cpi_33) & (df["CPI"] <= cpi_66), "Inflation_Bin"] = "Moderate Inflation"
df.loc[df["CPI"] > cpi_66, "Inflation_Bin"] = "High Inflation"

df["Inflation_Bin"] = df["Inflation_Bin"].astype("category")
df["Inflation_Bin"].value_counts()

In [ ]:
# Preview the rows
df.head()

In [ ]:
# Bin unemployment into levels
unemp_33, unemp_66 = df["Unemployment"].quantile([0.33, 0.66])

df["Unemployment_Bin"] = "Low Unemployment"
df.loc[(df["Unemployment"] > unemp_33) & (df["Unemployment"] <= unemp_66), "Unemployment_Bin"] = "Moderate Unemployment"
df.loc[df["Unemployment"] > unemp_66, "Unemployment_Bin"] = "High Unemployment"

df["Unemployment_Bin"] = df["Unemployment_Bin"].astype("category")
df["Unemployment_Bin"].value_counts()

In [ ]:
# Preview the rows
df.head()

In [ ]:
# Bin fuel price into levels
fuel_33, fuel_66 = df["Fuel_Price"].quantile([0.33, 0.66])

df["FuelPrice_Bin"] = "Low Fuel Price"
df.loc[(df["Fuel_Price"] > fuel_33) & (df["Fuel_Price"] <= fuel_66), "FuelPrice_Bin"] = "Medium Fuel Price"
df.loc[df["Fuel_Price"] > fuel_66, "FuelPrice_Bin"] = "High Fuel Price"

df["FuelPrice_Bin"] = df["FuelPrice_Bin"].astype("category")
df["FuelPrice_Bin"].value_counts()

In [ ]:
# Preview the rows
df.head()

In [ ]:
# Bin store size into Small, Medium and Large
size_33, size_66 = df["Size"].quantile([0.33, 0.66])

df["StoreSize_Bin"] = "Small Store"
df.loc[(df["Size"] > size_33) & (df["Size"] <= size_66), "StoreSize_Bin"] = "Medium Store"
df.loc[df["Size"] > size_66, "StoreSize_Bin"] = "Large Store"

df["StoreSize_Bin"] = df["StoreSize_Bin"].astype("category")
df["StoreSize_Bin"].value_counts()

In [ ]:
# Preview the rows
df.head()

In [ ]:
# Tag each holiday week by name
df["Holiday_Type"] = "Non-Holiday"

super_bowl = pd.to_datetime([
    "2018-02-08", "2019-02-12", "2020-02-11", "2021-02-10"
])

labor_day = pd.to_datetime([
    "2018-09-06", "2019-09-10", "2020-09-09", "2021-09-07"
])

thanksgiving = pd.to_datetime([
    "2018-11-29", "2019-11-26", "2020-11-25", "2021-11-23"
])

christmas = pd.to_datetime([
    "2018-12-27", "2019-12-31", "2020-12-30", "2021-12-28"
])

df.loc[df["Date"].isin(super_bowl), "Holiday_Type"] = "Super Bowl"
df.loc[df["Date"].isin(labor_day), "Holiday_Type"] = "Labor Day"
df.loc[df["Date"].isin(thanksgiving), "Holiday_Type"] = "Thanksgiving"
df.loc[df["Date"].isin(christmas), "Holiday_Type"] = "Christmas"

df["Holiday_Type"] = df["Holiday_Type"].astype("category")
df["Holiday_Type"].value_counts()

In [ ]:
# Preview the rows
df.head()

In [ ]:
# Plot the counts for each engineered category
import plotly.graph_objects as go

dark_colorscale = [
    [0, "#800000"],  
    [0.25, "#8B0000"], 
    [0.5, "#000080"], 
    [0.75, "#00008B"],
    [1, "#4B0082"]    
]

bin_cols = ["Promotion_Bin", "Inflation_Bin", "Unemployment_Bin",
            "FuelPrice_Bin", "StoreSize_Bin", "Holiday_Type"]

for col in bin_cols:
    counts = df[col].value_counts().sort_index()  # Count occurrences
    
    fig = go.Figure(
        data=go.Bar(
            y=counts.index.astype(str),  # Categories on the y-axis
            x=counts.values,             # Counts on the x-axis
            text=counts.values,          # Show counts on bars
            textposition="auto",
            orientation="h",             # Horizontal orientation
            marker=dict(color=counts.values, colorscale=dark_colorscale)
        )
    )

    fig.update_layout(
        title=f"Distribution of {col}",
        xaxis_title="Count",
        yaxis_title=col,
        width=600,
        height=500
    )

    fig.show()

### **1.4 Correlation Between Variables**

In [ ]:
# Correlation heatmap of the numeric drivers
corr = df.drop(["Date", "Year", "Month", "Week"], axis=1).corr(numeric_only=True)

fig = go.Figure(
    data=go.Heatmap(
        z=corr.values,  # 2D array of correlation values
        x=corr.columns,  # x-axis (Feature 2)
        y=corr.columns,  # y-axis (Feature 1)
        colorscale="RdBu",  # Red-blue colormap
        zmid=0,  # Center the color scale at 0
        text=corr.round(2).values,  # show correlation values
        texttemplate="%{text}",  # format text display
        hovertemplate="Feature 1: %{y}<br>Feature 2: %{x}<br>Correlation: %{z:.2f}<extra></extra>"
    )
)

fig.update_layout(
    title="Correlation Matrix: Weekly Retail Sales Drivers",
    width=800,
    height=800
)

fig.show()

### **1.6 KDE / Distribution Plots (Continuous Variables)**

In [ ]:
# KDE density plots for the key continuous variables
import numpy as np
from scipy.stats import gaussian_kde
import plotly.graph_objects as go

continuous_cols = ["Weekly_Sales", "Fuel_Price", "CPI", "Unemployment"]

colors = ["darkred", "darkblue", "darkgreen", "indigo"]  # You can customize more

for i, col in enumerate(continuous_cols):
    data = df[col].dropna()  # Drop missing values

    kde = gaussian_kde(data)
    x = np.linspace(data.min(), data.max(), 200)
    y = kde(x)

    fig = go.Figure(
        data=go.Scatter(
            x=x,
            y=y,
            mode='lines',
            fill='tozeroy',
            opacity=0.5,
            line=dict(color=colors[i % len(colors)]),  # Assign color from list
            name=f'{col} Density'
        ),
        layout=go.Layout(
            title=f'{col} Distribution (KDE)',
            xaxis_title=col,
            yaxis_title='Density',
            width=800,
            height=450
        )
    )
    fig.show()

### **2.1 Univariate Analysis – Weekly_Sales vs Variables**

In [ ]:
# Median weekly sales by each single category
import plotly.express as px

cols = ["Promotion_Bin", "Inflation_Bin", "Unemployment_Bin",
        "FuelPrice_Bin", "StoreSize_Bin", "Holiday_Type", "Type"]

for col in cols:
    grouped = (
        df.groupby(col)["Weekly_Sales"]
        .median()
        .reset_index()
        .assign(MedianSales=lambda d: d["Weekly_Sales"].round(0))  
    )

    grouped[col] = grouped[col].astype(str)

    fig = px.bar(
        grouped,
        x=col,
        y="MedianSales",
        text=grouped["MedianSales"].apply(lambda v: f"{v:,.0f}"),  
        color=col,
        color_discrete_sequence=px.colors.qualitative.Set3,  
        title=f"Median Weekly Sales by {col}",
        labels={"MedianSales": "Median Weekly Sales", col: col}
    )

    fig.update_layout(
        showlegend=False,  # remove legend
        yaxis=dict(title="Median Weekly Sales"),
        xaxis=dict(type="category", categoryorder="category ascending"),
        template="plotly_white"
    )

    fig.show()

### **2.2 Identifying Patterns Weekly_Sales Vs. Two Variable- (Bivariate Analysis)**

In [ ]:
# Median weekly sales across every pair of categories
bins = ["Promotion_Bin", "Inflation_Bin", "Unemployment_Bin",
        "FuelPrice_Bin", "StoreSize_Bin", "Holiday_Type", "Type"]

for primary_bin in bins:
    secondary_bins = [b for b in bins if b != primary_bin]
    
    for secondary_bin in secondary_bins:
        grouped = (
            df.groupby([primary_bin, secondary_bin])["Weekly_Sales"]
            .median()  # changed from mean() to median()
            .reset_index()
            .assign(AvgSales=lambda d: d["Weekly_Sales"].round(0))
        )
        
        secondary_categories = grouped[secondary_bin].unique()
        colors = px.colors.qualitative.Set2
        color_map = {cat: colors[i % len(colors)] for i, cat in enumerate(secondary_categories)}

        fig = go.Figure()
        
        for cat in secondary_categories:
            df_cat = grouped[grouped[secondary_bin] == cat]
            fig.add_trace(
                go.Bar(
                    x=df_cat[primary_bin].astype(str),
                    y=df_cat["AvgSales"],
                    name=str(cat),
                    text=df_cat["AvgSales"].apply(lambda v: f"{v:,.0f}"),
                    textposition="outside",
                    marker_color=color_map[cat]
                )
            )
        
        fig.update_layout(
            title=f"Weekly Sales by {primary_bin} and {secondary_bin}",
            xaxis_title=primary_bin,
            yaxis_title="Average Weekly Sales",
            barmode="group",
            template="plotly_white",
            showlegend=True,
            width=900,
            height=500
        )
        
        fig.show()

### **4.3.  Identifying Patterns Weekly_Sales Vs. Three Variables- (Multivariate Analysis)**

In [ ]:
# Median sales by promotion, holiday and store size
import plotly.express as px

df_plot = df.dropna(subset=["Promotion_Bin", "Holiday_Type", "StoreSize_Bin", "Weekly_Sales"]).copy()

df_plot["Promotion_Bin"] = df_plot["Promotion_Bin"].astype(str)
df_plot["Holiday_Type"] = df_plot["Holiday_Type"].astype(str)
df_plot["StoreSize_Bin"] = df_plot["StoreSize_Bin"].astype(str)

grouped = df_plot.groupby(["StoreSize_Bin", "Promotion_Bin", "Holiday_Type"], as_index=False)["Weekly_Sales"].median()
grouped["MedianSales"] = grouped["Weekly_Sales"].round(0)

fig = px.bar(
    grouped,
    x="Promotion_Bin",
    y="MedianSales",
    color="Holiday_Type",
    facet_row="StoreSize_Bin",  # separate row for each StoreSize_Bin
    barmode="group",
    text=grouped["MedianSales"].astype(str),
    labels={
        "MedianSales": "Median Weekly Sales",
        "Promotion_Bin": "Promotion Bin",
        "Holiday_Type": "Holiday Type",
        "StoreSize_Bin": "Store Size Bin"
    },
    title="Median Weekly Sales by Promotion_Bin, Holiday_Type, and StoreSize_Bin",
    height=900,
    width=1100,
    color_discrete_sequence=px.colors.qualitative.Set2  # bright color palette
)

for trace in fig.data:
    trace.textposition = ["inside" if y < 100 else "outside" for y in trace.y]
    trace.textfont.color = ["white" if y < 100 else "black" for y in trace.y]

fig.update_layout(
    template="plotly_white",
    showlegend=True,
    font=dict(size=14),
    bargap=0.25
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

fig.show()

In [ ]:
# Median sales by promotion, holiday and store type
import plotly.express as px

df_plot = df.dropna(subset=["Promotion_Bin", "Holiday_Type", "Type", "Weekly_Sales"]).copy()

df_plot["Promotion_Bin"] = df_plot["Promotion_Bin"].astype(str)
df_plot["Holiday_Type"] = df_plot["Holiday_Type"].astype(str)
df_plot["Type"] = df_plot["Type"].astype(str)

grouped = df_plot.groupby(["Type", "Promotion_Bin", "Holiday_Type"], as_index=False)["Weekly_Sales"].median()
grouped["MedianSales"] = grouped["Weekly_Sales"].round(0)

fig = px.bar(
    grouped,
    x="Promotion_Bin",
    y="MedianSales",
    color="Holiday_Type",
    facet_row="Type",  # separate row for each Type
    barmode="group",
    text=grouped["MedianSales"].astype(str),
    labels={
        "MedianSales": "Median Weekly Sales",
        "Promotion_Bin": "Promotion Bin",
        "Holiday_Type": "Holiday Type",
        "Type": "Store Type"
    },
    title="Median Weekly Sales by Promotion_Bin, Holiday_Type, and Type",
    height=900,
    width=1080,
    color_discrete_sequence=px.colors.qualitative.Set2  # bright colors
)

for trace in fig.data:
    trace.textposition = ["inside" if y < 100 else "outside" for y in trace.y]
    trace.textfont.color = ["white" if y < 100 else "black" for y in trace.y]

fig.update_layout(
    template="plotly_white",
    showlegend=True,
    font=dict(size=14),
    bargap=0.25
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

fig.show()

In [ ]:
# Median sales by promotion, inflation and unemployment
import plotly.express as px

df_plot = df.dropna(subset=["Promotion_Bin", "Inflation_Bin", "Unemployment_Bin", "Weekly_Sales"]).copy()

df_plot["Promotion_Bin"] = df_plot["Promotion_Bin"].astype(str)
df_plot["Inflation_Bin"] = df_plot["Inflation_Bin"].astype(str)
df_plot["Unemployment_Bin"] = df_plot["Unemployment_Bin"].astype(str)

grouped = df_plot.groupby(["Unemployment_Bin", "Promotion_Bin", "Inflation_Bin"], as_index=False)["Weekly_Sales"].median()
grouped["MedianSales"] = grouped["Weekly_Sales"].round(0)

fig = px.bar(
    grouped,
    x="Promotion_Bin",
    y="MedianSales",
    color="Inflation_Bin",
    facet_row="Unemployment_Bin",  # separate row for each Unemployment_Bin
    barmode="group",
    text=grouped["MedianSales"].astype(str),
    labels={
        "MedianSales": "Median Weekly Sales",
        "Promotion_Bin": "Promotion Bin",
        "Inflation_Bin": "Inflation Bin",
        "Unemployment_Bin": "Unemployment Bin"
    },
    title="Median Weekly Sales by Promotion_Bin, Inflation_Bin, and Unemployment_Bin",
    height=900,
    width=1100,
    color_discrete_sequence=px.colors.qualitative.Set2  # bright colors
)

for trace in fig.data:
    trace.textposition = ["inside" if y < 100 else "outside" for y in trace.y]
    trace.textfont.color = ["white" if y < 100 else "black" for y in trace.y]

fig.update_layout(
    template="plotly_white",
    showlegend=True,
    font=dict(size=14),
    bargap=0.25
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

fig.show()

In [ ]:
# Median sales by fuel price, holiday and store type
import plotly.express as px

df_plot = df.dropna(subset=["FuelPrice_Bin", "Holiday_Type", "Type", "Weekly_Sales"]).copy()

df_plot["FuelPrice_Bin"] = df_plot["FuelPrice_Bin"].astype(str)
df_plot["Holiday_Type"] = df_plot["Holiday_Type"].astype(str)
df_plot["Type"] = df_plot["Type"].astype(str)

grouped = df_plot.groupby(["Type", "FuelPrice_Bin", "Holiday_Type"], as_index=False)["Weekly_Sales"].median()
grouped["MedianSales"] = grouped["Weekly_Sales"].round(0)

fig = px.bar(
    grouped,
    x="FuelPrice_Bin",
    y="MedianSales",
    color="Holiday_Type",
    facet_row="Type",  # separate row for each Type
    barmode="group",
    text=grouped["MedianSales"].astype(str),
    labels={
        "MedianSales": "Median Weekly Sales",
        "FuelPrice_Bin": "Fuel Price Bin",
        "Holiday_Type": "Holiday Type",
        "Type": "Store Type"
    },
    title="Median Weekly Sales by FuelPrice_Bin, Holiday_Type, and Type",
    height=900,
    width=1100,
    color_discrete_sequence=px.colors.qualitative.Set2  # bright colors
)

for trace in fig.data:
    trace.textposition = ["inside" if y < 100 else "outside" for y in trace.y]
    trace.textfont.color = ["white" if y < 100 else "black" for y in trace.y]

fig.update_layout(
    template="plotly_white",
    showlegend=True,
    font=dict(size=14),
    bargap=0.25
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

fig.show()

### **4.3.  Time and Seasonlity Analysis**

In [ ]:
# Median weekly sales by month, one line per year
import plotly.express as px
import pandas as pd

monthly_sales = (
    df.groupby(['Year', 'Month'])['Weekly_Sales']
    .median()
    .reset_index()
    .rename(columns={'Weekly_Sales': 'MedianWeeklySales'})
)

fig = px.line(
    monthly_sales,
    x='Month',
    y='MedianWeeklySales',
    color='Year',
    markers=True,
    title='Median Weekly Sales by Month',
    labels={'MedianWeeklySales': 'Median Weekly Sales', 'Month': 'Month'}
)

fig.update_layout(
    xaxis=dict(tickmode='linear'),
    template='plotly_white',
    width=900,
    height=500
)

fig.show()

In [ ]:
# Median weekly sales by month, aggregated across all years
monthly_sales_bar = (
    df.groupby('Month')['Weekly_Sales']
    .median()
    .reset_index()
    .rename(columns={'Weekly_Sales': 'MedianWeeklySales'})
)

fig = px.bar(
    monthly_sales_bar,
    x='Month',
    y='MedianWeeklySales',
    text='MedianWeeklySales',
    title='Median Weekly Sales by Month Aggregate Across All Years',
    labels={'MedianWeeklySales': 'Median Weekly Sales', 'Month': 'Month'},
    color='MedianWeeklySales',
    color_continuous_scale='Purples'
)

fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(template='plotly_white', width=900, height=500)

fig.show()

In [ ]:
# Year-on-year median weekly sales
import plotly.express as px
import pandas as pd

yearly_sales = (
    df.groupby('Year')['Weekly_Sales']
    .median()  # You can change to .mean() to compare both perspectives
    .reset_index()
    .rename(columns={'Weekly_Sales': 'MedianWeeklySales'})
)

fig = px.line(
    yearly_sales,
    x='Year',
    y='MedianWeeklySales',
    markers=True,
    title='Year-on-Year Median Weekly Sales',
    labels={'MedianWeeklySales': 'Median Weekly Sales'}
)

fig.update_layout(
    template='plotly_white',
    width=900,
    height=500
)

fig.show()

In [ ]:
# Preview the rows
df.head()

### ***Machine Learning***

In [ ]:
# Engineer lag, rolling-mean and group-average features
numerical_cols = ['Size', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',"Promotional_Discount"]
df['Lag_1'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(1)
df['Lag_2'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(2)
df['Rolling_Mean_4'] = df.groupby(['Store','Dept'])['Weekly_Sales'].transform(lambda x: x.rolling(4).mean())
df['Store_Mean'] = df.groupby('Store')['Weekly_Sales'].transform('mean')
df['Dept_Mean'] = df.groupby('Dept')['Weekly_Sales'].transform('mean')

df['Month_Mean'] = df.groupby(['Year','Month'])['Weekly_Sales'].transform('mean')

In [ ]:
# Drop rows left without lag or rolling values
df = df.dropna(subset=['Lag_1','Lag_2','Rolling_Mean_4'])

In [ ]:
# Preview the rows
df.head()

In [ ]:
# Assemble the final feature list and target
numerical_cols_extended = numerical_cols + ['Lag_1','Lag_2','Rolling_Mean_4','Store_Mean','Dept_Mean','Month_Mean']

target_col = 'Weekly_Sales'

all_features = numerical_cols_extended

In [ ]:
# Split train and test by date at the 2020 cutoff
df = df.sort_values(by=['Store','Dept','Date'])

train_cutoff = '2020-12-31'

train_df = df[df['Date'] <= train_cutoff].copy()
test_df = df[df['Date'] > train_cutoff].copy()

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)

In [ ]:
# Build the feature matrices and target vectors
target_col = 'Weekly_Sales'
X_train = train_df[all_features]
y_train = train_df[target_col]

X_test = test_df[all_features]
y_test = test_df[target_col]

In [ ]:
# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Score a fitted model on train and test (R2, MSE, RMSE, MAE)
def evaluate_model(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    metrics = {
        'R2_train': r2_score(y_train, y_train_pred),
        'R2_test': r2_score(y_test, y_test_pred),
        'MSE_train': mean_squared_error(y_train, y_train_pred),
        'MSE_test': mean_squared_error(y_test, y_test_pred),
        'RMSE_train': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'RMSE_test': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'MAE_train': mean_absolute_error(y_train, y_train_pred),
        'MAE_test': mean_absolute_error(y_test, y_test_pred)
    }
    
    return metrics

In [ ]:
# Fit and evaluate linear regression
linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train)
lin_metrics = evaluate_model(linreg, X_train_scaled, y_train, X_test_scaled, y_test)

print("Linear Regression Metrics:")
print(pd.DataFrame(lin_metrics, index=[0]).T)

In [ ]:
# Fit and evaluate ridge regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
ridge_metrics = evaluate_model(ridge, X_train_scaled, y_train, X_test_scaled, y_test)

print("\nRidge Regression Metrics:")
print(pd.DataFrame(ridge_metrics, index=[0]).T)

In [ ]:
# Fit and evaluate lasso regression
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
lasso_metrics = evaluate_model(lasso, X_train_scaled, y_train, X_test_scaled, y_test)

print("\nLasso Regression Metrics:")
print(pd.DataFrame(lasso_metrics, index=[0]).T)

In [ ]:
# Fit and evaluate elastic net regression
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)
elastic_metrics = evaluate_model(elastic, X_train_scaled, y_train, X_test_scaled, y_test)

print("\nElastic Net Metrics:")
print(pd.DataFrame(elastic_metrics, index=[0]).T)

In [ ]:
# Compare the coefficients across the four models
coef_df = pd.DataFrame({
    'Feature': ['Size', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment','Promotional_Discount', 'Lag_1','Lag_2','Rolling_Mean_4','Store_Mean','Dept_Mean','Month_Mean'],
    'Linear': linreg.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
    'ElasticNet': elastic.coef_
})

print("\nComparison of Coefficients:")
print(coef_df)

In [ ]:
# Plot predicted vs actual weekly sales for every model
y_pred_lin = linreg.predict(X_test_scaled)
y_pred_ridge = ridge.predict(X_test_scaled)
y_pred_lasso = lasso.predict(X_test_scaled)
y_pred_elastic = elastic.predict(X_test_scaled)

plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred_lin, label='Linear', alpha=0.7)
plt.scatter(y_test, y_pred_ridge, label='Ridge', alpha=0.7)
plt.scatter(y_test, y_pred_lasso, label='Lasso', alpha=0.7)
plt.scatter(y_test, y_pred_elastic, label='ElasticNet', alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Weekly Sales')
plt.ylabel('Predicted Weekly Sales')
plt.title('Predicted vs Actual Weekly Sales')
plt.legend()
plt.show()